# yt-gpu-worker: warm HTTP server (FUTURE USE, optional)

Keeps FLUX.2 klein and SDXL-Lightning loaded on the GPUs and answers HTTP requests. **Studio's Kaggle batch flow does not use this notebook.**

**Before you run it**
1. Notebook settings: Accelerator = **GPU T4 x2**, **Internet = On**, keep the notebook **Private**.
2. Add Kaggle Secret `WORKER_TOKEN` (any long random string) and attach it to this notebook. Only if you set `TUNNEL = "ngrok"`: also `NGROK_AUTHTOKEN` and optionally `NGROK_DOMAIN`.
3. Run the cells in order. The last cell blocks and prints one log line per request, plus a heartbeat every minute.

**To stop:** press Stop on the last cell (or Session options, Stop session), or call `POST /v1/shutdown`. The session cap below is a safety net, not a substitute for stopping it yourself. An idle GPU session still uses your weekly quota.

In [ ]:
# ===== SETTINGS =====
REPO_URL = "https://github.com/DHIRAJSHEWALE007/yt-gpu-worker"
WORKER_REF = "main"            # branch or tag that contains http_worker/ (pin a tag once you have one)
GPU_ROLES = [(0, "flux2_klein"), (1, "sdxl_lightning")]   # (gpu index, engine); FLUX + SDXL do not fit on one T4
DEFAULT_ENGINE = "flux2_klein"
OUTPUT_FORMAT = "webp"         # webp | jpeg | png
TUNNEL = "none"                # none | ngrok | cloudflared  (keep 'none' until you decide to expose it)
PORT = 8000
MAX_SESSION_MINUTES = 360      # hard cap, 0 = off
IDLE_TIMEOUT_MINUTES = 0       # 0 = run until you stop it; e.g. 20 = stop when no request arrives for 20 min
SELFTEST_ON_START = True       # generates one small image per engine after loading

In [ ]:
!nvidia-smi
!free -h
!python --version

In [ ]:
import os, subprocess, sys
WORK = "/tmp/yt-gpu-worker"          # /tmp, NOT /kaggle/working (everything in /kaggle/working becomes kernel output)
if not os.path.isdir(WORK):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", WORKER_REF, REPO_URL, WORK], check=True)
os.chdir(WORK)
sys.path.insert(0, WORK)
assert os.path.isdir("http_worker"), "this ref has no http_worker/ folder; copy it into the repo and push first"

# never let pip touch torch/torchvision: Kaggle's preinstalled build is the tested one
lines = [l for l in open("requirements-gpu.txt").read().splitlines()
         if l.strip() and not l.lower().startswith(("torch", "#"))]
open("/tmp/req-notorch.txt", "w").write("\n".join(lines) + "\n")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "/tmp/req-notorch.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "fastapi", "uvicorn", "pyyaml", "pillow"], check=True)
if TUNNEL == "ngrok":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyngrok"], check=True)
print("setup done")

In [ ]:
from kaggle_secrets import UserSecretsClient
_sec = UserSecretsClient()

def _secret(name, required=False):
    try:
        return _sec.get_secret(name)
    except Exception:
        if required:
            raise RuntimeError(f"Kaggle Secret {name} is missing or not attached to this notebook")
        return None

os.environ["WORKER_TOKEN"] = _secret("WORKER_TOKEN", required=True)
if TUNNEL == "ngrok":
    os.environ["NGROK_AUTHTOKEN"] = _secret("NGROK_AUTHTOKEN", required=True)
    if _secret("NGROK_DOMAIN"):
        os.environ["NGROK_DOMAIN"] = _secret("NGROK_DOMAIN")
# lengths only, never the values
print({k: len(os.environ[k]) for k in ("WORKER_TOKEN", "NGROK_AUTHTOKEN", "NGROK_DOMAIN") if k in os.environ})

In [ ]:
import yaml
cfg = {
    "host": "127.0.0.1", "port": PORT,
    "roles": [{"gpu": g, "engine": e} for g, e in GPU_ROLES],
    "default_engine": DEFAULT_ENGINE, "output_format": OUTPUT_FORMAT, "image_quality": 92,
    "output_dir": "/tmp/http_worker_out",
    "max_session_minutes": MAX_SESSION_MINUTES, "idle_timeout_minutes": IDLE_TIMEOUT_MINUTES,
    "heartbeat_seconds": 60, "tunnel": TUNNEL, "selftest_on_start": SELFTEST_ON_START,
}
yaml.safe_dump(cfg, open("/tmp/http_worker_config.yaml", "w"))
print(open("/tmp/http_worker_config.yaml").read())

In [ ]:
# Start the server. This cell BLOCKS and streams logs until you stop it. Look for "[READY]" and "[SELFTEST] ... OK".
os.environ["PYTHONUNBUFFERED"] = "1"
!python -m http_worker.server --config /tmp/http_worker_config.yaml

### After the server has stopped
Nothing is left running. If you want it again, run all cells again (models are downloaded and loaded each session).

### Connecting Studio later
Studio's custom source `openai_images` or `studio_http_worker` needs: the public URL (only if you enabled a tunnel), the same `WORKER_TOKEN`, and a model name (`flux2_klein` or `sdxl_lightning`). See `http_worker/README.md`.